# OLAP Operations on a PM4Moodle Event Log

**Drill-down · Roll-up · Unfold · Fold**

This notebook takes the OCEL 2.0 log that **PM4Moodle extracts from Moodle** and applies the four
OLAP operations from:

> S. Khayatbashi, N. Miri, A. Jalali. *OLAP Operations for Object-Centric Process Mining.* CAiSE Forum 2025.

These operations are available directly in `pm4py` (≥ 2.7.23), which cites this paper — no extra library needed.

| Operation | What it changes | Inverse |
|---|---|---|
| `ocel_drill_down` | splits an **object type** by an attribute | `ocel_roll_up` |
| `ocel_unfold` | splits an **event type** by a related object type | `ocel_fold` |

**Requirements:** `pip install pm4py` and Graphviz installed on the system.

In [ ]:
import warnings
import pandas as pd
import pm4py

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

print("pm4py version:", pm4py.__version__)

---
## Step 0 · Load the log that PM4Moodle extracted

This is the real OCEL 2.0 export produced by PM4Moodle from a Moodle instance.

In [ ]:
import os

# PM4Moodle's exported OCEL 2.0 log
CANDIDATES = ["ocel2.0.json", "test_dataset/ocel2.0.json", "../test_dataset/ocel2.0.json"]
OCEL_PATH = next((p for p in CANDIDATES if os.path.exists(p)), CANDIDATES[0])
print("reading:", os.path.abspath(OCEL_PATH))

ocel = pm4py.read_ocel2_json(OCEL_PATH)
ocel

---
## Step 1 · Which object type should we drill down on?

Two signals tell you a type is worth splitting:

1. **Domain knowledge** — the type is a *catch-all* that bundles things that behave differently.
2. **The model itself** — that type's events collapse into one dense node or a self-loop in the OC-DFG.

In Moodle, **`forum`** is exactly such a catch-all. Moodle stores announcement forums, per-student
forums and open discussion forums all in the same table, distinguished only by a `type` attribute —
even though students *cannot post* in an announcement forum at all.

This is the same situation as the paper's hospital example, where `Test` hides *ECG* vs *Blood*.

In [ ]:
forums = ocel.objects[ocel.objects["ocel:type"] == "forum"][["ocel:oid", "ocel:type", "type", "name"]]
print("The single object type 'forum' actually contains:")
print(forums["type"].value_counts().to_string())
forums

---
## Step 2 · The baseline OC-DFG — what PM4Moodle shows today

We focus on the forum perspective and its core lifecycle activities so the diagram stays readable.

In [ ]:
CORE_ACTIVITIES = [
    "create_forum",
    "view_forum",
    "subscribe_to_forum",
    "add_discussion",
    "upload_post",
    "set_grade",
]

base = pm4py.filter_ocel_object_types(ocel, ["forum"])
base = pm4py.filter_ocel_event_attribute(base, "ocel:activity", CORE_ACTIVITIES, positive=True)
base

In [ ]:
ocdfg_base = pm4py.discover_ocdfg(base)
pm4py.view_ocdfg(ocdfg_base, graph_title="0 — Baseline: one generic 'forum' object type", format="png")

### Reading the baseline

Everything is a single colour: one `forum` object type. We can see *that* forums get viewed, subscribed to,
posted in and graded — but **not which kind of forum does what**. `view_forum` is one dense hub node and
`set_grade` sits on a self-loop. The distinct behaviour of announcement vs. collaboration forums is
completely averaged away.

---
## Step 3 · Drill-down — split `forum` by its `type` attribute

`ocel_drill_down(ocel, object_type, object_attribute)` rewrites the object type into the tuple
`"(forum, <value>)"`.

In [ ]:
drilled = pm4py.ocel_drill_down(base, "forum", "type")

print("object types BEFORE:", sorted(base.objects["ocel:type"].unique()))
print("object types AFTER: ", sorted(drilled.objects["ocel:type"].unique()))

In [ ]:
ocdfg_drilled = pm4py.discover_ocdfg(drilled)
pm4py.view_ocdfg(ocdfg_drilled, graph_title="1 — Drill-down: forum split by type", format="png")

### What the drill-down reveals

Now each forum kind has its own colour, and three findings jump out that were invisible before:

- **`(forum, eachuser)`** carries the entire collaboration lifecycle — `subscribe_to_forum`,
  `add_discussion`, `upload_post`. This is where students actually work.
- **`(forum, news)`** only ever reaches `view_forum` and `set_grade` — consistent with announcement
  forums, where students can read but not post.
- **`(forum, general)`** is created and viewed, and then goes nowhere — these open discussion forums
  are set up but never used. That is an actionable course-design finding.

---
## Step 4 · Roll-up — the inverse operation

`ocel_roll_up` collapses the sub-types back to the parent type, returning us to the baseline.
This is what makes the granularity adjustable in *both* directions.

In [ ]:
rolled = pm4py.ocel_roll_up(drilled, "forum")

print("object types after roll-up:", sorted(rolled.objects["ocel:type"].unique()))
print("identical to the baseline log:",
      sorted(rolled.objects["ocel:type"].unique()) == sorted(base.objects["ocel:type"].unique()))

---
## Step 5 · Unfold — split the *event* types too

Drill-down only relabels **objects**. The activity nodes are still shared: `view_forum` is one box
that several forum types point into.

`ocel_unfold(ocel, event_type, object_type)` rewrites the activity into `"(event_type, object_type)"`,
which finally separates the *sequences* belonging to each forum kind.

In [ ]:
FORUM_TYPES = ["news", "eachuser", "general"]

unfolded = drilled
for activity in CORE_ACTIVITIES:
    for ftype in FORUM_TYPES:
        unfolded = pm4py.ocel_unfold(unfolded, activity, f"(forum, {ftype})")

print("activities after unfolding:")
for a in sorted(unfolded.events["ocel:activity"].unique()):
    print("  ", a)

In [ ]:
ocdfg_unfolded = pm4py.discover_ocdfg(unfolded)
pm4py.view_ocdfg(ocdfg_unfolded, graph_title="2 — Unfold: activities split per forum type", format="png")

### What the unfold reveals

The shared hub is gone. Each forum type now has its **own chain of activities**, so the three
behaviours can be read as separate processes instead of one averaged model — which is precisely
the effect shown in Fig. 1 of the paper.

---
## Step 6 · Fold — back to coarse activities

`ocel_fold` is the inverse of `unfold`, merging the activity labels back together.

In [ ]:
folded = unfolded
for activity in CORE_ACTIVITIES:
    for ftype in FORUM_TYPES:
        folded = pm4py.ocel_fold(folded, activity, f"(forum, {ftype})")

print("activities after folding:", sorted(folded.events["ocel:activity"].unique()))

---
## Summary

| Step | Operation | Granularity of the model |
|---|---|---|
| 0 | — | one `forum` type, one averaged process |
| 1 | `ocel_drill_down` | 3 forum sub-types, objects separated |
| 2 | `ocel_roll_up` | back to the baseline |
| 3 | `ocel_unfold` | activities separated per forum type |
| 4 | `ocel_fold` | back to coarse activities |

The analyst moves **up and down the granularity scale on the same extracted log**, without
re-extracting anything from Moodle — which is what makes these operations a natural fit for PM4Moodle.

### Trying another dimension

Every line below is a valid alternative to demo — just change the two arguments:

```python
pm4py.ocel_drill_down(ocel, "question", "qtype")   # multichoice / truefalse / shortanswer / ...
pm4py.ocel_drill_down(ocel, "course",   "format")  # topics / weekly / ...
pm4py.ocel_drill_down(ocel, "user",     "auth")    # manual / ldap / ...
```

---
### Optional · Export the diagrams as PNGs for slides

In [ ]:
import os

os.makedirs("olap_figures", exist_ok=True)
for name, g in [
    ("0_baseline", ocdfg_base),
    ("1_drilldown", ocdfg_drilled),
    ("2_unfold", ocdfg_unfolded),
]:
    path = os.path.join("olap_figures", f"{name}.png")
    pm4py.save_vis_ocdfg(g, path)
    print("saved", path)